<a href="https://colab.research.google.com/github/dhanushkumar-amk/BUILD-OWN-XYZ/blob/main/own_tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Basic whitespace splitting**

In [ ]:
def whitespace_tokenize(text):
    tokens = text.split()  # splits on any whitespace, ignores extra spaces
    return tokens

# Test
sample = "I love pizza"
print(whitespace_tokenize(sample))

['I', 'love', 'pizza']


# **Add punctuation handling**

In [ ]:
import re

def tokenize(text):
    # Separates words from punctuation using regex
    # \w+ matches word characters, [^\w\s] matches punctuation
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

# Test
sample = "I love pizza!"
print(tokenize(sample))
# Expected: ['I', 'love', 'pizza', '!']

['I', 'love', 'pizza', '!']


## **Add lowercasing/normalization**

In [ ]:
def tokenize(text):
    text = text.lower()  # normalize case first
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

# Test
sample1 = "Pizza is great."
sample2 = "I love pizza."
print(tokenize(sample1))
print(tokenize(sample2))
# Both "Pizza" and "pizza" now become "pizza"

['pizza', 'is', 'great', '.']
['i', 'love', 'pizza', '.']


## **Build vocabulary from multiple texts**

In [ ]:
def build_vocabulary(texts):
    vocab = set()
    for text in texts:
        tokens = tokenize(text)
        vocab.update(tokens)
    return vocab

# Test
corpus = [
    "I love pizza",
    "I love pasta too",
    "Pizza is great!"
]
vocab = build_vocabulary(corpus)
print(vocab)
print("Vocab size:", len(vocab))

{'love', 'pizza', 'great', '!', 'pasta', 'is', 'too', 'i'}
Vocab size: 8


## **Add special tokens + build token-to-ID mapping**

In [ ]:
def build_mappings(vocab):
    # Reserve special tokens first
    special_tokens = ["<PAD>", "<START>", "<END>", "<UNK>"]

    all_tokens = special_tokens + sorted(vocab)  # sorted for consistent ordering

    token_to_id = {token: idx for idx, token in enumerate(all_tokens)}
    return token_to_id

# Test
token_to_id = build_mappings(vocab)
print(token_to_id)

{'<PAD>': 0, '<START>': 1, '<END>': 2, '<UNK>': 3, '!': 4, 'great': 5, 'i': 6, 'is': 7, 'love': 8, 'pasta': 9, 'pizza': 10, 'too': 11}


## **Build ID-to-token mapping**

In [ ]:
def build_reverse_mapping(token_to_id):
    id_to_token = {idx: token for token, idx in token_to_id.items()}
    return id_to_token

# Test
id_to_token = build_reverse_mapping(token_to_id)
print(id_to_token)

{0: '<PAD>', 1: '<START>', 2: '<END>', 3: '<UNK>', 4: '!', 5: 'great', 6: 'i', 7: 'is', 8: 'love', 9: 'pasta', 10: 'pizza', 11: 'too'}


## **Full encoder with UNK handling**

In [ ]:
def encode(text, token_to_id):
    tokens = tokenize(text)
    ids = []
    for token in tokens:
        if token in token_to_id:
            ids.append(token_to_id[token])
        else:
            ids.append(token_to_id["<UNK>"])  # fallback for unknown words
    return ids

# Test with a known sentence
print(encode("I love pizza", token_to_id))

# Test with an unknown word
print(encode("I love sushi", token_to_id))
# "sushi" wasn't in the vocab, should map to <UNK> id

[6, 8, 10]
[6, 8, 3]


## **Decoder**

In [ ]:
def decode(ids, id_to_token):
    tokens = [id_to_token[i] for i in ids]
    return " ".join(tokens)

# Test round trip
original = "I love pizza"
encoded = encode(original, token_to_id)
decoded = decode(encoded, id_to_token)

print("Original:", original)
print("Encoded:", encoded)
print("Decoded:", decoded)

Original: I love pizza
Encoded: [6, 8, 10]
Decoded: i love pizza


## **Wrap with START/END tokens**

In [ ]:
def encode_with_special_tokens(text, token_to_id):
    ids = encode(text, token_to_id)
    ids = [token_to_id["<START>"]] + ids + [token_to_id["<END>"]]
    return ids

# Test
print(encode_with_special_tokens("I love pizza", token_to_id))

[1, 6, 8, 10, 2]


## **Padding for batching multiple sentences**

In [ ]:
def pad_sequences(sequences, pad_id):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padding_needed = max_len - len(seq)
        padded_seq = seq + [pad_id] * padding_needed
        padded.append(padded_seq)
    return padded

# Test with two different-length sentences
seq1 = encode_with_special_tokens("I love pizza", token_to_id)
seq2 = encode_with_special_tokens("hi", token_to_id)

padded = pad_sequences([seq1, seq2], token_to_id["<PAD>"])
for p in padded:
    print(p)

[1, 6, 8, 10, 2]
[1, 3, 2, 0, 0]
